In [8]:
# ============================================================
# ROBUST PATH FIX
# ============================================================
import os
from pathlib import Path

try:
    project_root = Path(__file__).parent.parent
except NameError:
    cwd = Path(os.getcwd())
    project_root = cwd.parent if cwd.name == "notebooks" else cwd

os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")
# ============================================================


import duckdb

con = duckdb.connect()
con.execute("INSTALL spatial; LOAD spatial;")

file_path = "data/processed/tracts_acs_wind_2024_expanded.parquet"

print("=" * 70)
print("Exploratory Analysis (Clean Working Version)")
print("=" * 70)

# 1. Overall comparison
print("\n--- 1. Socio-Economic Stats by Wind Farm Presence ---")
con.sql(f"""
    SELECT has_wind_farm,
           COUNT(*) AS tract_count,
           ROUND(AVG(total_population), 0) AS avg_population,
           ROUND(AVG(median_household_income), 0) AS avg_income,
           ROUND(AVG(median_home_value), 0) AS avg_home_value
    FROM '{file_path}'
    GROUP BY has_wind_farm
    ORDER BY has_wind_farm
""").show()

# 2. Turbine distribution
print("\n--- 2. Turbine Count Distribution (Tracts With Wind Farms) ---")
con.sql(f"""
    SELECT turbine_count,
           COUNT(*) AS tract_count,
           ROUND(AVG(median_household_income), 0) AS avg_income,
           ROUND(AVG(median_home_value), 0) AS avg_home_value
    FROM '{file_path}'
    WHERE has_wind_farm = 1
    GROUP BY turbine_count
    ORDER BY turbine_count
    LIMIT 15
""").show()

# 3. High vs Low (grouped by raw turbine_count)
print("\n--- 3. High vs Low Turbine Count Tracts ---")
con.sql(f"""
    SELECT 
        CASE 
            WHEN turbine_count >= 50 THEN 'High (50+)'
            WHEN turbine_count BETWEEN 10 AND 49 THEN 'Medium (10-49)'
            ELSE 'Low (1-9)'
        END AS turbine_intensity,
        COUNT(*) AS tract_count,
        ROUND(AVG(total_population), 0) AS avg_population,
        ROUND(AVG(median_household_income), 0) AS avg_income,
        ROUND(AVG(median_home_value), 0) AS avg_home_value
    FROM '{file_path}'
    WHERE has_wind_farm = 1
    GROUP BY turbine_count
    ORDER BY turbine_count DESC
""").show()

# 4. Income & Home Value by Turbine Count (Simplified)
print("\n--- 4. Income & Home Value by Turbine Count ---")
con.sql(f"""
    SELECT turbine_count,
           COUNT(*) AS tract_count,
           ROUND(AVG(median_household_income), 0) AS avg_income,
           ROUND(AVG(median_home_value), 0) AS avg_home_value
    FROM '{file_path}'
    GROUP BY turbine_count
    ORDER BY turbine_count
    LIMIT 20
""").show()

print("=" * 70)

Working directory set to: /Users/jakemammen/Library/CloudStorage/OneDrive-Personal/Desktop/windfarm_socioecon_geospatial_analysis
Exploratory Analysis (Clean Working Version)

--- 1. Socio-Economic Stats by Wind Farm Presence ---
┌───────────────┬─────────────┬────────────────┬────────────┬────────────────┐
│ has_wind_farm │ tract_count │ avg_population │ avg_income │ avg_home_value │
│     int64     │    int64    │     double     │   double   │     double     │
├───────────────┼─────────────┼────────────────┼────────────┼────────────────┤
│             0 │        8611 │         4220.0 │    79553.0 │       278526.0 │
│             1 │         374 │         2901.0 │    69709.0 │       162324.0 │
└───────────────┴─────────────┴────────────────┴────────────┴────────────────┘


--- 2. Turbine Count Distribution (Tracts With Wind Farms) ---
┌───────────────┬─────────────┬────────────┬────────────────┐
│ turbine_count │ tract_count │ avg_income │ avg_home_value │
│     int64     │    int64  